# PySR Physics Discovery for P80

This notebook searches for a symbolic equation for `log_pi_P80 = log(P80 / L_char)` using pure dimensionless features. It reuses `build_geometry` from `geometry.py`, uses `sin_angle`/`cos_angle` instead of raw angle, removes the power operator, runs a longer search, and optimizes a relative-error-style loss on the original `pi_P80` scale.

In [1]:
import pandas as pd
import numpy as np
from pysr import PySRRegressor

from geometry import build_geometry

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


## Load Data

Use the provided forward-prediction training files.

In [2]:
data_dir = "forward_prediction"

print("Loading data...")
df = pd.read_csv(f"{data_dir}/train.csv")
y_raw = pd.read_csv(f"{data_dir}/train_labels.csv")

print(df.shape, y_raw.shape)
df.head()

Loading data...
(2930, 8) (2930, 6)


,porosity,atmosphere,gravity,coupling,strength,shape_factor,energy,angle_rad
0,0.337215,0.781263,3.71,0.861258,1.305809,0.784028,3.826405,0.818303
1,0.058029,0.136205,1.62,0.561245,3.494501,0.922737,2.828754,1.193036
2,0.315632,0.774704,3.71,0.948860,1.366386,0.954922,3.068907,0.605872
3,0.033062,0.144204,1.62,0.713705,3.599419,0.932911,2.700574,1.073708
4,0.278207,0.414620,9.81,1.237205,1.996742,1.260855,3.484022,0.863568


## Build Dimensionless Features

`build_geometry` computes `L_char`, `pi_strength`, angle trig features, and threshold pi-groups. This run uses pure, unlogged dimensionless inputs, with `sin_angle` and `cos_angle` replacing raw `angle_rad`. The target is `log_pi_P80`, which makes the symbolic equation multiplicative in original physical space.

In [3]:
geom = build_geometry(df)

X = pd.DataFrame(index=df.index)
X["pi_strength"] = geom["pi_strength"]
X["coupling"] = geom["coupling"]
X["porosity"] = geom["porosity"]
X["shape_factor"] = geom["shape_factor"]
X["sin_angle"] = geom["sin_angle"]
X["cos_angle"] = geom["cos_angle"]

# Log dimensionless target: log_pi_P80 = log(P80 / L_char)
pi_p80 = y_raw["P80"] / geom["L_char"]
y = np.log(pi_p80)

print(X.shape, y.shape)
for idx, row in X.iterrows():
    features = ", ".join(f"{col}={row[col]:.6g}" for col in X.columns)
    print(f"row {idx}: X=({features}), y={y.loc[idx]:.6g}")

(2930, 6) (2930,)
row 0: X=(pi_strength=0.420294, coupling=0.861258, porosity=0.337215, shape_factor=0.784028, sin_angle=0.729987, cos_angle=0.683461), y=4.27401
row 1: X=(pi_strength=8.3696, coupling=0.561245, porosity=0.0580295, shape_factor=0.922737, sin_angle=0.929493, cos_angle=0.368839), y=4.95717
row 2: X=(pi_strength=0.467676, coupling=0.94886, porosity=0.315632, shape_factor=0.954922, sin_angle=0.569479, cos_angle=0.822006), y=4.62868
row 3: X=(pi_strength=8.35601, coupling=0.713705, porosity=0.0330622, shape_factor=0.932911, sin_angle=0.878975, cos_angle=0.476868), y=4.93959
row 4: X=(pi_strength=0.510284, coupling=1.2372, porosity=0.278207, shape_factor=1.26085, sin_angle=0.760166, cos_angle=0.649729), y=4.75468
row 5: X=(pi_strength=0.172072, coupling=0.632437, porosity=0.303917, shape_factor=0.813381, sin_angle=0.647969, cos_angle=0.761666), y=4.58099
row 6: X=(pi_strength=1.36719, coupling=0.741432, porosity=0.0772367, shape_factor=0.897727, sin_angle=0.884215, cos_angle=

## Configure PySR

This configuration uses a longer search (`niterations=300`), removes the power operator to avoid brittle nested-power formulas, and uses a custom loss that measures relative squared error after converting `log_pi_P80` predictions back to `pi_P80` with `exp`.

In [ ]:
model = PySRRegressor(
    niterations=800,
    binary_operators=["+", "-", "*", "/", "greater"],
    unary_operators=["exp", "log"],
    # prediction and target are log(pi_P80); this measures relative error in original pi_P80 space.
    elementwise_loss="loss(prediction, target) = ((exp(prediction) - exp(target)) / (abs(exp(target)) + 1.0e-6))^2",
    maxsize=40,
    parsimony=0.001,
    output_directory="pysr_outputs/p80_log_relative",
    procs="auto",
    random_state=42,
)

## Run Symbolic Regression

PySR uses Julia under the hood. The first run may take extra time while Julia packages initialize.

In [ ]:
print("Starting evolution... check the notebook/terminal output for live updates.")
model.fit(X, y)

Starting evolution... check the notebook/terminal output for live updates.


/Users/wg/Documents/AIExperiment/Boom-Challenge-Datasets/.venv/lib/python3.14/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
/Users/wg/Documents/AIExperiment/Boom-Challenge-Datasets/.venv/lib/python3.14/site-packages/pysr/sr.py:1873: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
[ Info: Started!


## Inspect Results

In [ ]:
print("=" * 50)
print("Best discovered equation:")
print("=" * 50)
print(f"log_pi_P80 = {model.sympy()}")
print("\nFor original scale: pi_P80 = exp(log_pi_P80).")
print("All candidate equations are available in model.equations_ and PySR outputs under pysr_outputs/p80_log_relative.")

model.equations_

Best discovered equation:
log_pi_P80 = 4.7013664 - (-1)*0.65850705/(pi_strength + porosity/(-0.09421674))

For original scale: pi_P80 = exp(log_pi_P80).
All candidate equations are available in model.equations_ and PySR outputs under pysr_outputs/p80_log_relative.


,complexity,loss,equation,score,sympy_format,lambda_format
0,1,0.116166,4.578814,0.000000,4.57881400000000,PySRFunction(X=>4.57881400000000)
1,3,0.074435,5.20653 - cos_angle,0.222547,5.20653 - cos_angle,PySRFunction(X=>5.20653 - cos_angle)
2,5,0.070342,4.9329886 - (porosity * 1.7085401),0.028280,4.9329886 - 1.7085401*porosity,PySRFunction(X=>4.9329886 - 1.7085401*porosity)
3,6,0.068475,5.8316793 - exp(porosity * coupling),0.026898,5.8316793 - exp(coupling*porosity),PySRFunction(X=>5.8316793 - exp(coupling*poros...
4,7,0.062855,(0.30269387 / coupling) + (4.3965044 - porosity),0.085639,-porosity + 4.3965044 + 0.30269387/coupling,PySRFunction(X=>-porosity + 4.3965044 + 0.3026...
5,8,0.062837,(5.200903 - exp(-0.72284967 / coupling)) - por...,0.000278,-porosity + 5.200903 - exp(-0.72284967/coupling),PySRFunction(X=>-porosity + 5.200903 - exp(-0....
6,9,0.053701,4.7013664 - (-0.65850705 / (pi_strength + (por...,0.157124,4.7013664 - (-1)*0.65850705/(pi_strength + por...,PySRFunction(X=>4.7013664 - (-1)*0.65850705/(p...
7,11,0.047042,4.054181 - ((porosity - sin_angle) / ((pi_stre...,0.066194,4.054181 - (porosity - sin_angle)/(coupling + ...,PySRFunction(X=>4.054181 - (porosity - sin_ang...
8,13,0.046263,3.7965267 - (-1.100114 / ((porosity / 0.397832...,0.008352,3.7965267 - (-1)*1.100114/(coupling + pi_stren...,PySRFunction(X=>3.7965267 - (-1)*1.100114/(cou...
9,14,0.045270,(1.0537004 / ((((pi_strength * 0.035284683) + ...,0.021682,3.7853255 + 1.0537004/(coupling + (pi_strength...,PySRFunction(X=>3.7853255 + 1.0537004/(couplin...
